In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

np.random.seed(42)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def dsigmoid_from_a(a):
    return a * (1.0 - a)

class TwoLayerNN:
    def __init__(self, input_dim, hidden_dim, lr=0.5, epochs=4000):
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(1.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, 1) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, 1))
        self.lr = lr
        self.epochs = epochs
        self.losses = []

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    @staticmethod
    def bce_loss(y, yhat):
        eps = 1e-12
        yhat_c = np.clip(yhat, eps, 1.0 - eps)
        return -np.mean(y * np.log(yhat_c) + (1.0 - y) * np.log(1.0 - yhat_c))

    def compute_grads(self, X, y):
        m = X.shape[0]
        yhat = self.forward(X)
        # dL/dz2 = (yhat - y) / m for BCE + sigmoid
        dz2 = (yhat - y) / m
        dW2 = self.a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * dsigmoid_from_a(self.a1)
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)
        return dW1, db1, dW2, db2

    def fit(self, X, y):
        y = y.reshape(-1, 1)
        self.losses = []
        for _ in range(self.epochs):
            yhat = self.forward(X)
            self.losses.append(self.bce_loss(y, yhat))
            dW1, db1, dW2, db2 = self.compute_grads(X, y)
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
        return self

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int).ravel()

def plot_boundary(ax, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180))
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = model.forward(grid).reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=25, cmap="RdYlBu", alpha=0.75)
    ax.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=1.5)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="crimson", edgecolors="k", s=30, label="Class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="royalblue", edgecolors="k", s=30, label="Class 1")
    ax.set_title(title)
    ax.legend(fontsize=8)

# 1. XOR-style dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

nn_xor = TwoLayerNN(input_dim=2, hidden_dim=4, lr=1.5, epochs=5000).fit(X_xor, y_xor)
print("XOR predictions:", nn_xor.predict(X_xor), "| Target:", y_xor)

# Numerical gradient check on W1[0, 1] using XOR
eps = 1e-6
dW1_ana, _, _, _ = nn_xor.compute_grads(X_xor, y_xor.reshape(-1, 1))
nn_xor.W1[0, 1] += eps
loss_plus = nn_xor.bce_loss(y_xor.reshape(-1, 1), nn_xor.forward(X_xor))
nn_xor.W1[0, 1] -= 2 * eps
loss_minus = nn_xor.bce_loss(y_xor.reshape(-1, 1), nn_xor.forward(X_xor))
nn_xor.W1[0, 1] += eps
grad_num = (loss_plus - loss_minus) / (2 * eps)
print(f"Gradient check W1[0,1] -> Analytical: {dW1_ana[0,1]:.8f}, Numerical: {grad_num:.8f}, Diff: {abs(dW1_ana[0,1]-grad_num):.2e}")

# 2. Two-moons dataset
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)
nn_moons = TwoLayerNN(input_dim=2, hidden_dim=8, lr=0.8, epochs=4000).fit(X_moons, y_moons)
acc_moons = np.mean(nn_moons.predict(X_moons) == y_moons)
print(f"Two-Moons accuracy: {acc_moons*100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].plot(nn_xor.losses, lw=1.8, label="XOR Loss")
axes[0].plot(nn_moons.losses, lw=1.8, label="Two-Moons Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Training Loss vs. Epochs")
axes[0].legend()

plot_boundary(axes[1], nn_xor, X_xor, y_xor, "XOR Non-Linear Boundary")
plot_boundary(axes[2], nn_moons, X_moons, y_moons, f"Two-Moons Boundary (Acc: {acc_moons*100:.1f}%)")

plt.tight_layout()
plt.savefig("exp08_backprop_2layer.png", dpi=150, bbox_inches="tight")
plt.show()